# 1b · Re-train TSV on the Instruct model (one-model demo)

After notebooks 1–2, TSV's steering vector is still trained on the **base** model while SEP+HalluShift run on **-Instruct** — so the live demo would need TWO 8B models loaded at once (OOMs a 12 GB GPU; load/unload per question is unusable). This re-fits the tiny TSV head on the **Instruct** model so all three detectors share ONE model and the demo scores any question in seconds.

**Cheap (~0.1 GPU-hr):** reuses the answers + labels already in `data/triviaqa_fused.parquet` (no re-generation). Overwrites `best_checkpoint_retrained.pt` (now Instruct-trained) and the `tsv_margin` column, then refreshes the fusion. Run cell 1 (GPU) then cell 2 (CPU).

In [ ]:
import os, sys
os.environ.setdefault('HF_HOME', r'D:/LLAMA CACHE/huggingface')  # reuse local LLaMA cache
sys.path.insert(0, os.path.abspath(os.path.join('..', 'hallking')))
import warnings; warnings.filterwarnings('ignore')
DATASET='triviaqa'; EPOCHS_TSV=40
cmd=[sys.executable, os.path.join('..','tools','retrain_tsv_instruct.py'),
     '--dataset',DATASET,'--epochs_tsv',str(EPOCHS_TSV)]
import subprocess
_env = {**os.environ, 'PYTHONUNBUFFERED': '1'}
print('running:', ' '.join(cmd), flush=True)
_p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                      text=True, bufsize=1, env=_env)
for _line in _p.stdout:
    print(_line, end='', flush=True)
if _p.wait() != 0:
    raise RuntimeError(f'subprocess failed (rc={_p.returncode})')


### Refresh the honest OOF eval + fusion on the new Instruct-TSV margins
Re-runs notebook 2's evaluation so `models/fusion_<ds>_oof.pkl` matches the new TSV. (CPU, ~5 min)

In [ ]:
import os, sys
os.environ.setdefault('HF_HOME', r'D:/LLAMA CACHE/huggingface')  # reuse local LLaMA cache
sys.path.insert(0, os.path.abspath(os.path.join('..', 'hallking')))
import warnings; warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.join('..','tools'))
from honest_eval import evaluate
DATASET='triviaqa'
results, oof = evaluate(DATASET)
results